In [1]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.6 MB/s eta 0:00:00


In [2]:
"""
RSNA Knee -- LLM-based label extraction, KAGGLE/TRANSFORMERS VERSION.

Same extraction logic as the Ollama version, but using Hugging Face
transformers directly -- natively supported on Kaggle (pre-installed,
CUDA-ready), no separate background service needed like Ollama requires.

Setup: transformers and torch are already on Kaggle. Just make sure your
notebook has GPU enabled (Settings -> Accelerator -> GPU T4).
"""

import json
import re
import time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

CFG_TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

MODEL_NAME = "Qwen/Qwen3-8B"

# 4-bit quantization -- keeps the full 8B model within a T4's 16GB VRAM.
# Full bf16 weights alone would need ~16GB (8B params x 2 bytes), leaving no
# room for generation. 4-bit compresses that to ~4GB, real headroom left over.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {MODEL_NAME} in 4-bit... (first run downloads the weights, needs internet -- fine for this prep step)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=quantization_config, device_map="auto"
)
print("Model loaded.")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def load_reports(csv_path: str, report_column: str):
    """Unchanged."""
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=[report_column])
    return df


def extract_json_block(raw: str) -> str:
    """Unchanged -- same regex fix from before. Also robustly handles Qwen3's
    optional 'thinking' text before the real answer, since it just grabs the
    {...} block wherever it actually is, ignoring everything around it."""
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        return match.group(0)
    raise ValueError("No JSON object found in response")


def build_extraction_prompt(report_text: str) -> str:
    """UNCHANGED from your validated prompt -- same wording, same fixes."""
    targets_list = ", ".join(CFG_TARGETS)
    prompt = (
        f"Analyze the following radiology report and extract the status of each of these 12 findings: {targets_list}.\n"
        f"Report: {report_text}\n"
        "For each finding, use ONLY strict literal reading of the report text:\n"
        "- 1 if the report explicitly states this finding is present\n"
        "- 0 if the report explicitly states this finding is absent or normal\n"
        "- null if the report does not explicitly mention this specific finding, "
        "even if a related or nearby structure is discussed\n"
        "Do not infer a finding is absent just because a similar or nearby structure was mentioned instead.\n"
        "Respond with ONLY the JSON object — no explanation, no notes, no markdown code fences, "
        "no text before or after the JSON."
    )
    return prompt


def extract_labels_for_report(report_text: str, max_retries: int = 3) -> dict:
    prompt = build_extraction_prompt(report_text)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,  # Qwen3-specific: skip internal reasoning, go straight to the answer
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    for attempt in range(max_retries):
        try:
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs, max_new_tokens=300, do_sample=False,
                )
            new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
            raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            cleaned = extract_json_block(raw)
            return json.loads(cleaned)
        except (json.JSONDecodeError, Exception) as e:
            if attempt == max_retries - 1:
                print(f"Failed after {max_retries} attempts: {e}")
                print(f"RAW OUTPUT WAS: {repr(raw[:500])}")
                return {t: None for t in CFG_TARGETS}
            time.sleep(1)
    return {t: None for t in CFG_TARGETS}


def run_extraction(df: pd.DataFrame, report_column: str, uid_column: str, output_path: str):
    """FIX 2 applied here: real sequential counter via enumerate(), not the
    original dataframe index from .iterrows(), which breaks once df is a
    sliced subset with non-sequential original indices."""
    results = {}
    for count, (i, row) in enumerate(df.iterrows()):
        uid = row[uid_column]
        report = row[report_column]
        labels = extract_labels_for_report(report)
        results[uid] = labels
        if count % 50 == 0:
            print(f"Processed {count}/{len(df)}")
            with open(output_path, "w") as f:
                json.dump(results, f, indent=2)
    with open(output_path, "w") as f:
        json.dump(results, f, indent=2)
    return results


def validate_against_hard_labels(extracted: dict, hard_labeled_df: pd.DataFrame, uid_column: str):
    """Unchanged -- same validation logic you already trust."""
    hard_labeled_df = hard_labeled_df.set_index(uid_column)
    extracted_df = pd.DataFrame.from_dict(extracted, orient="index")
    extracted_df = extracted_df.reindex(hard_labeled_df.index)
    comparison_df = hard_labeled_df.join(extracted_df, lsuffix="_hard", rsuffix="_extracted")

    agreement = {}
    counts = {}
    for target in CFG_TARGETS:
        hard_col = f"{target}_hard"
        extracted_col = f"{target}_extracted"
        valid_rows = comparison_df[extracted_col].notnull()
        matches = (comparison_df.loc[valid_rows, hard_col] == comparison_df.loc[valid_rows, extracted_col]).sum()
        total = valid_rows.sum()
        agreement[target] = matches / total if total > 0 else None
        counts[target] = total

    agreement_df = pd.DataFrame.from_dict(agreement, orient="index", columns=["agreement"])
    agreement_df["n"] = pd.Series(counts)
    print(agreement_df)
    overall_agreement = agreement_df["agreement"].mean()
    print(f"Overall agreement: {overall_agreement:.2f}")
    return overall_agreement


if __name__ == "__main__":
    REPORT_COLUMN = "Report"
    UID_COLUMN = "StudyInstanceUID"

    DATA_PATH = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
    full_df = load_reports(DATA_PATH, REPORT_COLUMN)
    hard_labeled = pd.read_csv(DATA_PATH).dropna(subset=CFG_TARGETS)

    RUN_VALIDATION = False

    if RUN_VALIDATION:
        validation_subset = full_df[full_df[UID_COLUMN].isin(hard_labeled[UID_COLUMN])]
        print(f"Running validation pass on {len(validation_subset)} known-labeled studies first...")
        validation_extracted = run_extraction(validation_subset, REPORT_COLUMN, UID_COLUMN, "validation_check.json")
        validate_against_hard_labels(validation_extracted, hard_labeled, UID_COLUMN)
        print("\nReview the agreement numbers above against ~0.78 (your original Claude-based run).")
        print("If comparable, set RUN_VALIDATION = False and rerun to move on to full extraction below.")

    else:
        # ── STEP B: The full, multi-session extraction ──────────────────
        # FIX 3: point this at the MERGED session1+2 file (3601 reports),
        # not session 1 alone -- update the path to match whatever you
        # actually named the dataset once you upload combined_extracted_labels.json
        PRIOR_SESSION_OUTPUT = "/kaggle/input/datasets/chiragggg/extracted-labels-session1/combined_extracted_labels.json"
        OUTPUT_PATH = "/kaggle/working/extracted_labels.json"
        REPORTS_PER_SESSION = 1600

        if PRIOR_SESSION_OUTPUT is not None:
            with open(PRIOR_SESSION_OUTPUT, "r") as f:
                already_done = json.load(f)
            print(f"Resuming from prior session: {len(already_done)} reports already extracted.")
        else:
            already_done = {}
            print("Starting fresh -- no prior session output provided.")

        remaining_df = full_df[~full_df[UID_COLUMN].isin(already_done.keys())]
        this_session_df = remaining_df.iloc[:REPORTS_PER_SESSION]  # only ever process this capped slice
        print(f"Remaining overall: {len(remaining_df)} of {len(full_df)} total reports.")
        print(f"Processing this session: {len(this_session_df)} (capped at {REPORTS_PER_SESSION})")

        if len(this_session_df) == 0:
            print("Nothing left to extract -- all reports already done in a prior session.")
        else:
            # FIX 1 applied here: this_session_df, NOT remaining_df --
            # this was the actual bug that caused session 2 to ignore the cap
            new_results = run_extraction(this_session_df, REPORT_COLUMN, UID_COLUMN, "extracted_labels_this_session.json")
            combined = {**already_done, **new_results}
            with open(OUTPUT_PATH, "w") as f:
                json.dump(combined, f, indent=2)
            print(f"Saved combined total: {len(combined)} reports to {OUTPUT_PATH}")
            print(f"Still remaining after this session: {len(full_df) - len(combined)}")


Loading Qwen/Qwen3-8B in 4-bit... (first run downloads the weights, needs internet -- fine for this prep step)


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded.
GPU memory allocated: 1.57 GB


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Resuming from prior session: 3601 reports already extracted.
Remaining overall: 806 of 4407 total reports.
Processing this session: 806 (capped at 1600)
Processed 0/806
Processed 50/806
Processed 100/806
Processed 150/806
Processed 200/806
Processed 250/806
Processed 300/806
Processed 350/806
Processed 400/806
Processed 450/806
Processed 500/806
Processed 550/806
Processed 600/806
Processed 650/806
Processed 700/806
Processed 750/806
Processed 800/806
Saved combined total: 4407 reports to /kaggle/working/extracted_labels.json
Still remaining after this session: 0


In [3]:
print(model.hf_device_map)

{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 1, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}
